In [2]:
# =========================================================
# 1. IMPORT LIBRARIES
# =========================================================

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

import warnings
warnings.filterwarnings("ignore")

In [3]:
# =========================================================
# 2. LOAD PROCESSED DATA
# =========================================================

X_train = pd.read_csv(
    "../results/X_train.csv"
)

X_test = pd.read_csv(
    "../results/X_test.csv"
)

y_train = pd.read_csv(
    "../results/y_train.csv"
)["target"]

y_test = pd.read_csv(
    "../results/y_test.csv"
)["target"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (4000, 133)
X_test: (1000, 133)
y_train: (4000,)
y_test: (1000,)


In [6]:
# =========================================================
# 3. LOGISTIC REGRESSION
# =========================================================

pipe_lr = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "lr",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
])

param_grid_lr = {
    "lr__C": [0.01, 0.1, 1, 10],
    "lr__solver": [
        "lbfgs",
        "newton-cg"
    ]
}

lr_grid = GridSearchCV(
    estimator=pipe_lr,
    param_grid=param_grid_lr,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

print("Training Logistic Regression...")

lr_grid.fit(
    X_train,
    y_train
)

best_lr = lr_grid.best_estimator_

import joblib

joblib.dump(best_lr, "../models/logistic_regression.pkl")

print("Logistic Regression model saved!")

y_pred_lr = best_lr.predict(
    X_test
)

print("\nBest Parameters:")
print(
    lr_grid.best_params_
)

Training Logistic Regression...


FileNotFoundError: [Errno 2] No such file or directory: '../models/logistic_regression.pkl'

In [5]:
# =========================================================
# 4. LOGISTIC REGRESSION EVALUATION
# =========================================================

class_names = [
    "Fair",
    "Good",
    "Poor"
]

print(
    "\nLogistic Regression Report:"
)

print(
    classification_report(
        y_test,
        y_pred_lr,
        target_names=class_names
    )
)


Logistic Regression Report:
              precision    recall  f1-score   support

        Fair       0.96      0.66      0.78       722
        Good       0.74      0.93      0.82       255
        Poor       0.11      0.87      0.20        23

    accuracy                           0.73      1000
   macro avg       0.60      0.82      0.60      1000
weighted avg       0.88      0.73      0.78      1000

